# C05 — RAG Pipeline (Retrieval-Augmented Generation)

Pipeline completo de RAG: indexação de documentos, recuperação por similaridade e geração de respostas.

In [ ]:
%pip install anthropic sentence-transformers faiss-cpu python-dotenv -q

## 1. Base de conhecimento (corpus)

In [ ]:
CORPUS = [
    "RAG (Retrieval-Augmented Generation) combina busca de documentos com geração de texto por LLMs.",
    "O processo de RAG tem duas etapas: recuperação (retrieval) e geração (generation).",
    "Na etapa de recuperação, documentos relevantes são buscados num índice vetorial usando embeddings.",
    "Na etapa de geração, o LLM recebe os documentos recuperados como contexto para responder a pergunta.",
    "FAISS é uma biblioteca do Facebook para busca eficiente de vizinhos mais próximos em espaços vetoriais.",
    "Embeddings são representações numéricas de texto que capturam significado semântico.",
    "LangChain é um framework popular para construir pipelines RAG e agentes com LLMs.",
    "A qualidade do RAG depende da qualidade dos embeddings e da relevância dos documentos recuperados.",
    "Chunking é a divisão de documentos longos em trechos menores antes de indexar.",
    "Re-ranking melhora a precisão do RAG ao reordenar os documentos recuperados por relevância.",
]

## 2. Indexação — criação do índice vetorial

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
corpus_embs = embedder.encode(CORPUS, normalize_embeddings=True).astype(np.float32)

dim = corpus_embs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(corpus_embs)

print(f"Documentos indexados: {index.ntotal} | Dimensão: {dim}")

## 3. Recuperação — função de busca

In [ ]:
def recuperar(query: str, k: int = 3) -> list[str]:
    q_emb = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    distances, indices = index.search(q_emb, k)
    return [CORPUS[i] for i in indices[0]]

# Teste
query_teste = "Como funciona a etapa de busca no RAG?"
docs = recuperar(query_teste)
print(f"Query: {query_teste}\n")
for i, d in enumerate(docs, 1):
    print(f"  [{i}] {d}")

## 4. Geração — resposta aumentada por contexto

In [ ]:
import os
import anthropic
from dotenv import load_dotenv

load_dotenv()
llm = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def rag(query: str, k: int = 3) -> str:
    docs = recuperar(query, k)
    contexto = "\n".join(f"- {d}" for d in docs)

    system = (
        "Você é um assistente especialista. "
        "Use SOMENTE as informações fornecidas no contexto para responder. "
        "Se a informação não estiver no contexto, diga que não sabe."
    )
    user = f"Contexto:\n{contexto}\n\nPergunta: {query}"

    resp = llm.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        system=system,
        messages=[{"role": "user", "content": user}]
    )
    return resp.content[0].text

## 5. Pipeline completo em ação

In [ ]:
perguntas = [
    "O que é RAG e para que serve?",
    "O que é chunking em RAG?",
    "Qual é a diferença entre retrieval e generation?",
]

for pergunta in perguntas:
    print(f"Pergunta: {pergunta}")
    print(f"Resposta: {rag(pergunta)}")
    print("-" * 70)

## 6. Avaliação simples (faithfulness check)

In [ ]:
def avaliar_fidelidade(query: str, resposta: str) -> str:
    docs = recuperar(query)
    contexto = "\n".join(f"- {d}" for d in docs)

    prompt = f"""
Dado o contexto abaixo, avalie se a resposta é fiel ao contexto.
Responda apenas: FIEL, PARCIALMENTE FIEL ou INFIEL, seguido de uma justificativa de 1 frase.

Contexto:
{contexto}

Resposta avaliada:
{resposta}
"""
    return llm.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=128,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text


q = "O que é RAG e para que serve?"
r = rag(q)
print(f"Pergunta : {q}")
print(f"Resposta : {r}")
print(f"Avaliação: {avaliar_fidelidade(q, r)}")